# 0e · **transfer eval** — GPU 2장 (2 seed × 5 rep = **10 run**)

150k 체크포인트를 **5회 반복 × 500 에피소드** 평가한다. 이 창은 **seed 2개**만 맡는다.

| | |
|---|---|
| 평가 대상 | **150k 체크포인트 하나** (best-ckpt 안 고름 → 모델 간 공정) |
| 반복 | **5회**, rep 마다 `--seed = 1000 + 100·rep` → env 초기상태가 달라짐 |
| 이 창 | **2 seed × 5 rep = 10 run** (× 500 에피소드 = 5,000) |
| 나머지 seed | `PART` 를 바꿔 **다른 창/노드**에서 → 합쳐서 4 seed × 5 rep = 20 run |

GPU 2장을 그대로 쓴다(동시에 2 run). 끝난 run 은 **자동 skip** → 중단/재실행 안전.

> rep 마다 seed 를 바꾸는 이유: 같은 seed 로 5번 돌리면 결정적이라 반복이 무의미하다.
> 학습 seed(모델 분산)와 rep(평가 분산)이 이렇게 분리된다.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import importlib, common_final as cf
importlib.reload(cf)

TASK = cf.SHORT_SIM        # ★ 'transfer'
PART = 'A'                 # ★ 이 창이 맡을 몫:  'A' = seed 0,1   /  'B' = seed 2,3
TAGS = cf.GROUP_OURS       # ['ours'].  cf.GROUP_ACM / cf.GROUP_BASELINE + ['act_te'] 등

SEEDS, GPUS = cf.part(PART)          # GPU 는 이 노드에 보이는 것으로 자동 배정
REPS = list(range(cf.EVAL_REPEATS))  # 5회
N_EP = cf.EVAL_N_EP                  # 500 에피소드

n_run = len(TAGS) * len(SEEDS) * len(REPS)
print('task :', TASK, '| PART:', PART)
print('모델 :', TAGS, '| seeds:', SEEDS, '| GPU:', GPUS)
print('eval : %s ckpt x %d rep x %d ep' % (f'{cf.CKPT_STEP:,}', len(REPS), N_EP))
print('run  :', n_run, f'(= 에피소드 {n_run * N_EP:,})')


## 1) 사전 확인 — 150k 체크포인트가 있는가
**X 가 있으면 eval 하지 말 것.** 150k 에 못 간 모델을 평가하면 학습량이 다른 것끼리 비교하게 된다.
(아직이면 `0r_resume_train` 으로 이어서 학습)


In [ ]:
ok = cf.print_ckpt_status([t for t in TAGS if t != 'act_te'], SEEDS, TASK)
print()
print('=>', 'eval 진행 가능' if ok else '⚠️ 학습 먼저 (0r_resume_train)')


## 2) 반복 eval — 5회 × 2 seed (GPU 2장, 동시 2 run)
이미 끝난 run 은 skip 하므로, 끊겨도 이 셀을 다시 실행하면 남은 것만 이어서 돈다.


In [ ]:
cf.run_repeat_evals(TAGS, SEEDS, REPS, task=TASK, gpus=GPUS, n_episodes=N_EP)


## 3) 결과 (이 창의 seed 만)
전체 표(4 seed pooled)는 두 창이 다 끝난 뒤 `09_report_sr` / `18_report_horizon`.


In [ ]:
rows = cf.sr_table(TAGS, SEEDS, REPS, task=TASK, n_episodes=N_EP)


## 4) rep 별로 뜯어보기 — 평가 분산 확인
같은 체크포인트를 5번 돌린 결과. 흩어짐이 크면 에피소드 수를 늘려야 한다는 신호.


In [ ]:
for t in TAGS:
    for s in SEEDS:
        vals = [cf.rep_sr(t, s, TASK, r) for r in REPS]
        got = [v for v in vals if v is not None]
        if not got:
            continue
        import statistics as st
        spread = st.pstdev(got) if len(got) > 1 else 0.0
        cells = '  '.join(f'rep{r}:{v:5.1f}' if v is not None else f'rep{r}:  -  '
                          for r, v in zip(REPS, vals))
        print(f'{t}/seed{s}   {cells}   | mean {st.fmean(got):5.1f}  std {spread:4.1f}')


## 다음
- 나머지 seed: `PART = 'B'` 로 바꿔 **다른 창/노드**에서 이 노트북을 다시 (합쳐서 20 run)
- 다른 모델: `TAGS = cf.GROUP_ACM` → `cf.GROUP_BASELINE + ['act_te']` 순으로 재실행
- 표·그림: `09_report_sr` · `10_report_jerk` · `18_report_horizon`

> 한 4-GPU 노드에서 A·B 를 동시에 띄울 때만 GPU 를 직접 나눌 것:
> `SEEDS, GPUS = cf.part('B', gpus=[2, 3])`
